# Wave 1 — Config / PromptManager / MemoryManager

**먼저 `00_setup.ipynb`를 실행하세요.**

## 1. Config 테스트

In [ ]:
%%ipytest
from app.factories.config import Config, LLMConfig, EmbeddingConfig, VectorDBConfig, PromptConfig, CONFIGS

def test_llmconfig_defaults():
    cfg = LLMConfig(provider='ollama', model_name='llama3')
    assert cfg.temperature == 0
    assert cfg.num_ctx == 8192
    assert cfg.base_url is None

def test_embedding_resolve_uses_own_url():
    cfg = EmbeddingConfig(base_url='http://embed:11434')
    assert cfg.resolve_base_url('http://llm:11434') == 'http://embed:11434'

def test_embedding_resolve_falls_back_to_llm_url():
    cfg = EmbeddingConfig()
    assert cfg.resolve_base_url('http://llm:11434') == 'http://llm:11434'

def test_embedding_resolve_falls_back_to_localhost():
    cfg = EmbeddingConfig()
    assert cfg.resolve_base_url(None) == 'http://localhost:11434'

def test_vectordb_defaults():
    cfg = VectorDBConfig()
    assert cfg.chunk_size == 800
    assert cfg.chunk_overlap == 150
    assert cfg.retrieval_k == 5
    assert cfg.score_threshold == 0.7

def test_configs_has_expected_keys():
    assert 'ollama_config' in CONFIGS
    assert 'openai_config' in CONFIGS

def test_configs_all_values_are_config_instances():
    for key, val in CONFIGS.items():
        assert isinstance(val, Config), f'{key} is not a Config instance'

UsageError: Cell magic `%%ipytest` not found.


## 2. PromptManager 테스트

In [ ]:
%%ipytest
import json, pytest
from app.core.prompt_manager import PromptManager

REGISTRY = {
    'tech_expert': {'persona': '기술 전문가입니다.', 'guide': '간결하게 답변하세요.'},
    'maintenance_expert': {'persona': '유지보수 전문가입니다.', 'guide': '안전 절차를 안내하세요.'},
}

@pytest.fixture
def pm(tmp_path):
    p = tmp_path / 'registry.json'
    p.write_text(json.dumps(REGISTRY, ensure_ascii=False), encoding='utf-8')
    return PromptManager(registry_path=str(p))

def test_loads_registry(tmp_path):
    p = tmp_path / 'registry.json'
    p.write_text(json.dumps(REGISTRY, ensure_ascii=False), encoding='utf-8')
    pm = PromptManager(registry_path=str(p))
    assert 'tech_expert' in pm.registry

def test_raises_if_file_not_found(tmp_path):
    with pytest.raises(FileNotFoundError):
        PromptManager(registry_path=str(tmp_path / 'nonexistent.json'))

def test_build_fills_template_fields(pm):
    result = pm.build('tech_expert', question='질문', history='히스토리', context='컨텍스트')
    assert '히스토리' in result
    assert '컨텍스트' in result
    assert '질문' in result

def test_build_uses_none_placeholder(pm):
    result = pm.build('tech_expert', question='Q')
    assert '없음' in result

def test_build_uses_custom_persona(pm):
    result = pm.build('tech_expert', question='Q', custom_persona='커스텀 페르소나')
    assert '커스텀 페르소나' in result
    assert '기술 전문가입니다.' not in result

def test_build_falls_back_to_tech_expert(pm):
    result = pm.build('nonexistent_id', question='Q')
    assert '기술 전문가입니다.' in result

def test_build_template_structure(pm):
    result = pm.build('tech_expert', question='Q')
    for section in ['[페르소나]', '[이전 대화]', '[참고 정보]', '[질문]', '[답변 지침]']:
        assert section in result

## 3. MemoryManager 테스트

In [ ]:
%%ipytest
from app.core.memory_manger import MemoryManager

def test_empty_session_returns_empty_string():
    mm = MemoryManager()
    assert mm.get_history('new_session') == ''

def test_save_creates_session():
    mm = MemoryManager()
    mm.save('s1', '질문', '답변')
    assert 's1' in mm.sessions

def test_history_contains_question_and_answer():
    mm = MemoryManager()
    mm.save('s1', '무엇인가요?', '이것입니다.')
    history = mm.get_history('s1')
    assert '무엇인가요?' in history
    assert '이것입니다.' in history

def test_window_eviction_respects_k():
    mm = MemoryManager(k=2)
    for i in range(3):
        mm.save('s1', f'Q{i}', f'A{i}')
    history = mm.get_history('s1')
    assert 'Q0' not in history
    assert 'Q2' in history

def test_clear_removes_session():
    mm = MemoryManager()
    mm.save('s1', 'Q', 'A')
    mm.clear('s1')
    assert 's1' not in mm.sessions

def test_clear_nonexistent_does_not_raise():
    mm = MemoryManager()
    mm.clear('does_not_exist')

def test_sessions_are_independent():
    mm = MemoryManager()
    mm.save('s1', 'Q1', 'A1')
    mm.save('s2', 'Q2', 'A2')
    assert 'Q2' not in mm.get_history('s1')
    assert 'Q1' not in mm.get_history('s2')